# 06 — roof plate / neural crest subclustering

**Feeds:** ED Fig 7

**Position in the chain:** run the numbered stages in order

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up. `PROJECT_ROOT`, which was the analysis directory, is now `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, in the same layout. `DEV_ROOT`, under which the notebook writes, is `$SCRNASEQ_RESULTS_ROOT/trunk_main_dev/` (default `scrnaseq/output/trunk_main_dev/`) instead of the analysis directory's `trunk_main_dev/`. The earlier trunk morph object from the Zenodo deposit is read from `$SCRNASEQ_INPUT_ROOT/legacy/results/intermediates/02_trunk_main/adata_morph_with_clusters.h5ad` (`SCRNASEQ_INPUT_ROOT`, default `data/scrnaseq_inputs/`).
2. The SMD z-score files are read from `scrnaseq/chain_inputs/trunk_main_dev/results/smd_runs/` (`PROJECT_ROOT / "trunk_main_dev" / "results"`) instead of `RESULTS_DIR`, where they sat in the original analysis directory.
3. Removed the `.to_clipboard()` 2 calls, which copied tables to the system clipboard for pasting into the Supplementary Data spreadsheet and fail on a machine without a clipboard. The expression before each call is unchanged.
4. The combined roof plate / neural crest subcluster labels come from one `sc.tl.leiden` call on the selected-gene graph (resolution 0.75, seed 5, `flavor="igraph"`, `n_iterations=-1`, as `03` calls it), with the cluster-id to name map written into the cell. The original read the same run's labels and names from files saved by a separate clustering-stability analysis that is not part of this repository. `sc.tl.leiden` writes the raw cluster ids to `obs["leiden_morph_rpnc_raw"]`. The cell then overwrites that column with the suffixed cluster names, as before. In the saved `meta.json`, `ensemble_default_label_source` holds that analysis's relative directory instead of an absolute path; the other strings naming the label source are unchanged.

No other line of code was changed.


# 06 - Roof Plate / Neural Crest Subclustering

Depends on: 01_preprocessing, 02_trunk_main artifacts.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root, scrnaseq_input_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)
# The earlier trunk morph object, fetched from the Zenodo deposit (see data/DOWNLOAD.md).
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO_ROOT)

import os
os.chdir(REPO_ROOT)


## Setup and Imports


In [ ]:
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

/opt/anaconda3/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-pa

In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:
from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under trunk_main_dev/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "trunk_main_dev"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
from src.trunk_morph_ref.aggregation import (
    cluster_averages_sparse_safe,
)
from src.trunk_morph_ref.correlation import (
    gene_corrcoef_sparse_safe,
)
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    contains_symbol,
    ensure_gene_id_index,
    filter_genes_sparse_safe as filter_genes,
    build_legacy_smd_score_table,
    install_scanpy_symbol_defaults,
    map_legacy_smd_scores_to_adata,
    normalize_unit_variance_sparse_safe as normalize_unit_variance,
    resolve_symbol_dict,
    resolve_symbols,
    symbols_missing,
    symbols_present,
    var_names_to_symbols,
)

# Use gene symbols for plot labels while keeping `var_names` as stable gene IDs.
install_scanpy_symbol_defaults(sc)


In [ ]:
from src.trunk_morph_ref.pipeline_io import (
    load_h5ad,
    load_npy,
    load_pickle,
    save_h5ad,
    save_json,
    save_npy,
    save_pickle,
    stage_dir,
)


In [ ]:
pre_path = stage_dir(RESULTS_DIR, '01_preprocessing')
trunk_path = stage_dir(RESULTS_DIR, '02_trunk_main')
adata_morph_raw = load_h5ad(pre_path / 'adata_morph_raw.h5ad')
adata_morph_with_clusters = load_h5ad(trunk_path / 'adata_morph_with_clusters.h5ad')
for _adata in [
    adata_morph_raw,
    adata_morph_with_clusters,
]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)

print('Loaded 01_preprocessing and 02_trunk_main intermediates')


Roof Plate / Neural Crest subclustering analysis workflow.

Current inputs come from active `trunk_main_dev` `01` + final `02` coarse labels. The notebook still maps canonical legacy SMD z-scores from `z_morph_nc4.03_000.npy` and `z_morph_rp4.03_000.npy` for the historical separate NC/RP analyses, but the primary fresh-SMD path is now the combined `Roof Plate + Neural Crest` neighborhood. Separate NC/RP matrices are retained only for historical continuity and comparison.


### Neural Crest pre-processing

In [ ]:
genes_cellcycle = [
    "MKI67",
    "CENPE",
    "SGO2",
    "KIF14",
    "PIF1",
    "NDC80",
    "CDCA8",
    "PLK1",
    "AURKA",
    "UBE2C",
    "ASPM",
    "TOP2A",
    "TPX2",
    "NUSAP1",
    "CDC20",
    "CKS2",
    "KPNA2",
    "TUBB4B",
    "DLGAP5",
    "BIRC5",
    "HMMR",
    "CCNB1",
    "ARL6IP1",
    "PTTG1",
    "UBE2S",
    "DUT",
    "HELLS",
    "CLSPN",
    "RRM2",
    "PCLAF",
    "TYMS",
    "KIF18B",
    "SMC4",
    "HIST1H4C",
    "DIAPH3",
    "RFC3",
    "MIR924HG",
    "MIS18BP1",
    "TUBA1C",
    "CDK1",
    "CENPA",
    # extra genes added after mesoderm subclustering, were not already in SMD gene list
    "KIF11",
    "ECT2",
    "KNL1",
    "NEK2",
    "CEP55",
    "PSRC1",
    "CDCA3",
    "CCNA2",
    "GTSE1",
    "CENPF",
    "CKS1B",
    "MELK",
    "CDCA5",
    # Added 4/8 during LPM sub-subclustering
    "MCM4",
    "HIST1H1E",
    "BRCA2",
    "TRIM66",
    "C1orf112",
    "POLD3",
    "KIF23",
    # Added 4/8 part 2
    "CDKAL1",
    "RAD51AP1",
    "RANBP1",
    "CDCA2",
    "CAND2",
    "PCLAF",
    "EIF5A",
    "SRSF2",
    "GINS1",
    "GINS2",
    # Added 4/9 Neuron subclustering
    "ncAPG2",
    "CCND1",
    "ATAD2",
    "UBE2T",
    "CENPK",
    "LAPTM4B",
    "CCND2",
    "MED13",
    "ZNF37A",
    "PTPN14",
    # 4/10 Neural Crest
    "ncAPG",
    "TTK",
    "KIF4A",
    "KIF18A",
    "CDKN3",
    "CEP70",
    "BRIP1",
    "SPC25",
    "KIFC1",
    "NSD2",
    "BUB1",
    "BUB1B",
    "ANP32E",
    "HMGB3",
    "PCNA",
    "CENPP",
    # 4/10 Roof plate
    "CCT5",
    "KIF15",
    "CSRP2",
    "ORC6",
    "HMGB2",
]

In [ ]:
# nc clusters
adata_morph_nc = adata_morph_raw.copy()

# Filter for cells more than 5k reads
nc_minimum_reads = 5000
sc.pp.filter_cells(adata_morph_nc, min_counts=nc_minimum_reads)

# Downsample such that each cell has 10k reads
sc.pp.downsample_counts(
    adata_morph_nc, counts_per_cell=nc_minimum_reads, replace=True, random_state=0
)


adata_morph_nc.obs["leiden_morph"] = adata_morph_with_clusters.obs["leiden_morph"].reindex(
    adata_morph_nc.obs_names
)
missing_cluster_labels = int(adata_morph_nc.obs["leiden_morph"].isna().sum())
if missing_cluster_labels:
    print(
        f"Dropping {missing_cluster_labels} cells not present in adata_morph_with_clusters "
        "before coarse-cluster subsetting."
    )
    adata_morph_nc = adata_morph_nc[adata_morph_nc.obs["leiden_morph"].notna()].copy()

In [ ]:
nc_clusters = ["Neural Crest"]

# Select cells that are in nc clusters
adata_morph_nc = adata_morph_nc[
    (adata_morph_nc.obs.leiden_morph.isin(nc_clusters).values)
]

# Log-transform read counts
sc.pp.log1p(adata_morph_nc)

# For SMD analysis:
# Filter for log-transformed genes with mean value greater than 0.05 and standard deviation greater than 0.05
adata_morph_nc_filtered = filter_genes(
    adata_morph_nc, mean_cutoff=0.05, std_cutoff=0.05
)
# Normalize each gene to have unit variance in expression
adata_morph_nc_filtered = normalize_unit_variance(adata_morph_nc_filtered)

In [ ]:
# Filter for coefficient of variation (std deviation / mean) > 1
# Since we have normalized to unit variance this is just a mean cutoff < 1
counts = adata_morph_nc_filtered.X
cv_mask = np.mean(counts, axis=0).transpose() < 1
adata_morph_nc_filtered = adata_morph_nc_filtered[:, cv_mask]

In [ ]:
# # Save pre-processed expression data to .npz file, for input to SMD on cluster
rpnc_stage_path = RESULTS_DIR / "intermediates" / "06_rp_nc_subclustering"
rpnc_stage_path.mkdir(parents=True, exist_ok=True)
legacy_nc_input_path = rpnc_stage_path / "morph_nc4.03.npz"
canonical_nc_input_path = rpnc_stage_path / "trunk_morph_nc__smd_input.npz"
for _path in [legacy_nc_input_path, canonical_nc_input_path]:
    scipy.sparse.save_npz(_path, adata_morph_nc_filtered.X)
recommended_n_sub_nc = int(round(0.8 * len(adata_morph_nc_filtered)))
save_json(
    {
        "stage": "06_rp_nc_subclustering",
        "subset_name": "neural_crest",
        "subset_clusters": nc_clusters,
        "minimum_reads": nc_minimum_reads,
        "legacy_smd_input_file": legacy_nc_input_path.name,
        "canonical_smd_input_file": canonical_nc_input_path.name,
        "n_cells_after_preprocessing": int(adata_morph_nc_filtered.n_obs),
        "n_genes_after_preprocessing": int(adata_morph_nc_filtered.n_vars),
        "recommended_n_sub": recommended_n_sub_nc,
        "status": "ready_for_legacy_compatible_smd_mapping",
    },
    rpnc_stage_path / "pre_smd_input_meta_nc.json",
)
print(
    f"{len(adata_morph_nc_filtered)} NC cells after pre-processing\n"
    f"{adata_morph_nc_filtered.shape[1]} genes after preprocessing\n"
    f"Recommended n_sub ~= {recommended_n_sub_nc}"
)


In [ ]:
# For downstream analysis after SMD:
# Only filter genes for non-zero mean and standard deviation:
adata_morph_nc_nofilter = adata_morph_nc.copy()
adata_morph_nc = filter_genes(adata_morph_nc, mean_cutoff=0, std_cutoff=0)
adata_morph_nc = normalize_unit_variance(adata_morph_nc)

### Neural Crest post-SMD processing

In [ ]:
# Load z-score file from morph_nc SMD output
legacy_nc_scores = build_legacy_smd_score_table(
    legacy_adata_path=SCRNASEQ_INPUT_ROOT
    / "legacy/results/intermediates/02_trunk_main/adata_morph_with_clusters.h5ad",
    legacy_cluster_col="leiden_morph",
    legacy_clusters=nc_clusters,
    zscore_path=PROJECT_ROOT / "z_morph_nc4.03_000.npy",
    score_col="z_score_morph_nc",
    raw_adata=adata_morph_raw,
    minimum_reads=nc_minimum_reads,
    mean_cutoff=0.05,
    std_cutoff=0.05,
    cv_mean_cutoff=1.0,
    random_state=0,
)
adata_morph_nc_filtered, nc_score_map_diag = map_legacy_smd_scores_to_adata(
    adata_morph_nc_filtered,
    legacy_nc_scores,
    score_col="z_score_morph_nc",
)
z_scores_morph_nc = adata_morph_nc_filtered.var["z_score_morph_nc"].to_numpy()
print(
    "Mapped legacy morph_nc z-scores to current filtered genes: "
    + f"{nc_score_map_diag['n_matched']} matched, "
    + f"{nc_score_map_diag['n_missing_in_current']} missing in current data, "
    + f"{nc_score_map_diag['n_legacy_only']} legacy-only genes ignored."
)

# Plot top morph_nc SMD z-scores
smd_cutoff = 2

plt.figure(figsize=(12, 4))
plt.hlines(smd_cutoff, -1, 2000, "r")  # z_score cutoff for genes: >= 2
plt.plot(sorted(z_scores_morph_nc)[::-1], "k.", markersize=5)
plt.yscale("log")
plt.ylim(0.01, 1.5 * z_scores_morph_nc.max())
plt.xlim(-1, 1000)
plt.ylabel("morph_nc z-score")
print(
    "\n"
    + str((z_scores_morph_nc > smd_cutoff).sum())
    + " morph_nc genes with z_score >= cutoff"
)
plt.show()

In [ ]:
# Filter for top morph_nc genes by z-score
adata_morph_nc_filtered_ = adata_morph_nc_filtered.copy()
adata_morph_nc_ = adata_morph_nc.copy()

# Select genes with z-score > cutoff
SMDmorph_ncgenes = list(
    adata_morph_nc_filtered_.var_names[
        adata_morph_nc_filtered_.var.z_score_morph_nc > smd_cutoff
    ]
)

manual_addgenes = []
manual_removegenes = []

# Resolve selected gene symbols to gene IDs before matrix subsetting.
SMDmorph_ncgenes = resolve_symbols(adata_morph_nc_, SMDmorph_ncgenes, strict=False, allow_missing=True)
# Resolve cell-cycle symbols to gene IDs before dropping them from clustering features.
cellcycle_gene_ids = set(resolve_symbols(adata_morph_nc_, genes_cellcycle, strict=False, allow_missing=True))
SMDmorph_ncgenes = [g for g in SMDmorph_ncgenes if g not in cellcycle_gene_ids]

adata_morph_nc_.var["z_score_morph_nc"] = None
for g in adata_morph_nc_filtered_.var_names:
    adata_morph_nc_.var.loc[g, "z_score_morph_nc"] = (
        adata_morph_nc_filtered_.var.loc[g, "z_score_morph_nc"]
    )
# Subset columns by resolved gene IDs to keep symbol mapping deterministic.
adata_morph_nc_SMD = adata_morph_nc_[:, resolve_symbols(adata_morph_nc_, SMDmorph_ncgenes, strict=False, allow_missing=True)]

print(
    str(len(adata_morph_nc_SMD.var))
    + " genes selected with morph_nc z > cutoff after cell-cycle removal"
)

In [ ]:
corr_morph_nc_gg = np.corrcoef(adata_morph_nc_SMD.X.todense().T)

cluster_gg_morph_nc = sns.clustermap(
    corr_morph_nc_gg,
    method="ward",
    metric="euclidean",
    figsize=(20, 20),
    cmap=batlow,
    vmin=-0.2,
    vmax=0.8,
    yticklabels=var_names_to_symbols(adata_morph_nc_SMD),
    xticklabels=var_names_to_symbols(adata_morph_nc_SMD),
)

In [ ]:
# Keep the 02-style simplified NC panel: z > cutoff only, with cell-cycle removal.
print("Skipping correlation-based gene pruning and manual gene overrides for neural crest; using the 02-style simplified panel.")


In [ ]:
corr_morph_nc_gg = gene_corrcoef_sparse_safe(adata_morph_nc_SMD.X)

cluster_gg_morph_nc = sns.clustermap(
    corr_morph_nc_gg,
    method="ward",
    metric="euclidean",
    figsize=(20, 20),
    cmap=batlow,
    vmin=-0.2,
    vmax=0.8,
    yticklabels=var_names_to_symbols(adata_morph_nc_SMD),
    xticklabels=var_names_to_symbols(adata_morph_nc_SMD),
)

#### Supp. Data 1 - SMD Neural Crest Z-scores

In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 1, "Gene markers and differential expression supporting trunk morph cell type annotations"
# Sheet 2, "SMD Z-scores"
# Columns J-K "Neural Crest Subclustering"
try:
    adata_morph_nc_SMD.var.z_score_morph_nc.sort_values(ascending=False)
except Exception as exc:
    print(f"Skipping NC clipboard export: {exc}")


### Neural Crest Clustering in SMD gene space

In [ ]:
# Cluster neural crest data independently

# Create a copy and normalize to uniform reads among morph cells
adata_morph_nc_SMD_ = adata_morph_nc_SMD.copy()


sc.pp.normalize_total(adata_morph_nc_SMD_)
try:
    sc.pp.neighbors(adata_morph_nc_SMD_, use_rep="X")
except:
    sc.pp.neighbors(adata_morph_nc_SMD_, use_rep="X")
sc.tl.umap(adata_morph_nc_SMD_, random_state=0)

# Leiden clustering
sc.tl.leiden(
    adata_morph_nc_SMD_,
    resolution=0.66,
    random_state=0,
    key_added="leiden_morph_nc",
    flavor="igraph",
    n_iterations=-1,
)


celltypes_morph_nc = {
    "1": "Neural Crest-Derived Immature Neurons",
    # NEUROG1, NEUROG2, TUBB3, ELAVL2, neuropeptides
    "4": "Migratory Cranial Neural Crest",
    # SOX10+, SNAI2 med, EDRNA has craniofacial defects, NPR3 regulates cranial ncs,
    "2": "Neural Crest-Derived Neurogenic Progenitors",
    # SOX2 high CDH2 HES5 but also SOX10, possible coclustering with non-nc neural progenitors from non-bead
    # Could be neural progenitors prior to pre-migratory cluster but why SOX10 expression?
    "0": "Pre-Migratory Neural Crest",
    # low SNAI2, SOX10-, high CDH2, ZFHX4, TPBG linked to EMT
    "3": "Early Migratory Neural Crest",
    # high SNAI2, high SOX10, low CDH2, higher collagen suggests chondrocyte-bias but not sure
}

adata_morph_nc_SMD_.obs["leiden_morph_nc"] = adata_morph_nc_SMD_.obs[
    "leiden_morph_nc"
].cat.rename_categories(celltypes_morph_nc)

# Copy cluster IDs to adata with all genes
adata_morph_nc_ = adata_morph_nc.copy()
adata_morph_nc_.obs["leiden_morph_nc"] = adata_morph_nc_SMD_.obs[
    "leiden_morph_nc"
]

adata_morph_nc_SMD_leiden = sc.get.aggregate(
    adata_morph_nc_SMD_,
    by="leiden_morph_nc",
    func=["count_nonzero", "mean", "sum", "var"],
    axis="obs",
)

corr_clcl_morph_nc = np.corrcoef(adata_morph_nc_SMD_leiden.layers["mean"])

with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": (300)}):
    cluster_clcl_morph_nc = sns.clustermap(
        corr_clcl_morph_nc,
        method="ward",
        metric="euclidean",
        figsize=(10, 10),
        cmap=batlow,
        # vmin=-0.2,
        # vmax=1,
        yticklabels=adata_morph_nc_SMD_leiden.obs.leiden_morph_nc.cat.categories,
        xticklabels=adata_morph_nc_SMD_leiden.obs.leiden_morph_nc.cat.categories,
    )
    cluster_clcl_morph_nc.fig.suptitle(
        "Cluster-cluster mean gene correlation among all expressed genes"
    )
    # plt.savefig("figures/cluster-cluster_correlation.pdf")

adata_morph_nc_.obs.leiden_morph_nc = (
    adata_morph_nc_.obs.leiden_morph_nc.cat.reorder_categories(
        adata_morph_nc_.obs.leiden_morph_nc.cat.categories[
            cluster_clcl_morph_nc.dendrogram_col.reordered_ind
        ]
    )
)

adata_morph_nc_SMD_.obs.leiden_morph_nc = (
    adata_morph_nc_SMD_.obs.leiden_morph_nc.cat.reorder_categories(
        adata_morph_nc_SMD_.obs.leiden_morph_nc.cat.categories[
            cluster_clcl_morph_nc.dendrogram_col.reordered_ind
        ]
    )
)

adata_morph_nc_SMD_ = adata_morph_nc_SMD_[
    adata_morph_nc_SMD_.obs.sort_values("leiden_morph_nc").index, :
]
adata_morph_nc_ = adata_morph_nc_[
    adata_morph_nc_.obs.sort_values("leiden_morph_nc").index, :
]

In [ ]:
celltypeorder = [
    "Pre-Migratory Neural Crest",
    "Early Migratory Neural Crest",
    "Migratory Cranial Neural Crest",
    "Neural Crest-Derived Neurogenic Progenitors",
    "Neural Crest-Derived Immature Neurons",
]
nc_series = adata_morph_nc_SMD_.obs["leiden_morph_nc"].cat.remove_unused_categories()
observed_celltypes = list(nc_series.cat.categories)
missing_expected = [ct for ct in celltypeorder if ct not in observed_celltypes]
unexpected_celltypes = [ct for ct in observed_celltypes if ct not in celltypeorder]
if missing_expected:
    print("NC categories missing from current partition:", missing_expected)
if unexpected_celltypes:
    print("Appending unexpected NC categories at end of ordering:", unexpected_celltypes)
ordered_celltypes = [ct for ct in celltypeorder if ct in observed_celltypes] + unexpected_celltypes
adata_morph_nc_SMD_.obs["leiden_morph_nc"] = nc_series.cat.reorder_categories(
    ordered_celltypes
)
adata_morph_nc_SMD_ = adata_morph_nc_SMD_[
    adata_morph_nc_SMD_.obs.leiden_morph_nc.sort_values().index, :
]

with plt.rc_context({"figure.dpi": (300)}):
    linkage_genes = scipy.cluster.hierarchy.linkage(
        adata_morph_nc_SMD_.X.todense().T,
        method="complete",
        metric="jensenshannon",
        optimal_ordering=True,
    )

    for t_dist in [
        0.61
    ]:  # np.linspace(0.4, 1, 61):  # t_dist 50 splits into one gene per cluster
        print(t_dist)
        adata_morph_nc_SMD_leiden.X = adata_morph_nc_SMD_leiden.layers["mean"]

        adata_morph_nc_SMD_.var["fcluster"] = scipy.cluster.hierarchy.fcluster(
            linkage_genes, t=t_dist, criterion="distance"
        )

        adata_morph_nc_SMD_fcluster = sc.get.aggregate(
            adata_morph_nc_SMD_,
            by="fcluster",
            func=["count_nonzero", "mean", "sum", "var"],
            axis="var",
        )

        adata_morph_nc_SMD_fcluster.X = adata_morph_nc_SMD_fcluster.layers["mean"]

        # Calculate top 100 DEGs per cluster, among all genes, by multinomial logistic regression
        sc.tl.rank_genes_groups(
            adata_morph_nc_SMD_fcluster,
            groupby="leiden_morph_nc",
            method="wilcoxon",
            rankby_abs=False,
            max_iter=1000,
            multi_class="multinomial",
        )
        DEGs_morph_fcluster = pd.DataFrame(
            adata_morph_nc_SMD_fcluster.uns["rank_genes_groups"]["names"]
        )
        DEGscores_morph_fcluster = pd.DataFrame(
            adata_morph_nc_SMD_fcluster.uns["rank_genes_groups"]["scores"]
        )

        for i in adata_morph_nc_SMD_fcluster.var.fcluster.values:
            adata_morph_nc_SMD_fcluster.var.loc[str(i), "leiden"] = (
                DEGscores_morph_fcluster[DEGs_morph_fcluster == str(i)]
                .max(axis=0)
                .idxmax()
            )
            adata_morph_nc_SMD_fcluster.var.loc[str(i), "score"] = (
                DEGscores_morph_fcluster[DEGs_morph_fcluster == str(i)]
                .max(axis=0)
                .max()
            )

        adata_morph_nc_SMD_fcluster.var["leiden"] = (
            adata_morph_nc_SMD_fcluster.var["leiden"].astype(
                pd.CategoricalDtype(categories=celltypeorder, ordered=True)
            )
        )

        print(
            [
                ct
                for ct in celltypeorder
                if ct
                not in set(list(adata_morph_nc_SMD_fcluster.var["leiden"].values))
            ]
        )
        # assert(set(adata_morph_SMD_fcluster.var["leiden"].values) == set(celltypeorder))

        adata_morph_nc_SMD_fcluster = adata_morph_nc_SMD_fcluster[
            :,
            adata_morph_nc_SMD_fcluster.var.sort_values(
                ["leiden", "score"], ascending=[True, False]
            ).index,
        ]

        clusterorder = list(adata_morph_nc_SMD_fcluster.var.index.astype("int32"))

        adata_morph_nc_SMD_.var.fcluster = (
            adata_morph_nc_SMD_.var.fcluster.astype(
                pd.CategoricalDtype(categories=clusterorder, ordered=True)
            )
        )

        adata_morph_nc_SMD_2 = adata_morph_nc_SMD_[
            :, adata_morph_nc_SMD_.var.fcluster.sort_values().index
        ]
        print(len(set(adata_morph_nc_SMD_2.var.fcluster)))
        sc.pl.heatmap(
            adata_morph_nc_SMD_2,
            var_names=adata_morph_nc_SMD_2.var_names,
            groupby="leiden_morph_nc",
            swap_axes=True,
            show_gene_labels=True,
            figsize=(10, 10),
            vmin=0,
            vmax=5,
            cmap=batlow,
        )

### Roof Plate pre-processing

In [ ]:
genes_cellcycle = [
    "MKI67",
    "CENPE",
    "SGO2",
    "KIF14",
    "PIF1",
    "NDC80",
    "CDCA8",
    "PLK1",
    "AURKA",
    "UBE2C",
    "ASPM",
    "TOP2A",
    "TPX2",
    "NUSAP1",
    "CDC20",
    "CKS2",
    "KPNA2",
    "TUBB4B",
    "DLGAP5",
    "BIRC5",
    "HMMR",
    "CCNB1",
    "ARL6IP1",
    "PTTG1",
    "UBE2S",
    "DUT",
    "HELLS",
    "CLSPN",
    "RRM2",
    "PCLAF",
    "TYMS",
    "KIF18B",
    "SMC4",
    "HIST1H4C",
    "DIAPH3",
    "RFC3",
    "MIR924HG",
    "MIS18BP1",
    "TUBA1C",
    "CDK1",
    "CENPA",
    # extra genes added after mesoderm subclustering, were not already in SMD gene list
    "KIF11",
    "ECT2",
    "KNL1",
    "NEK2",
    "CEP55",
    "PSRC1",
    "CDCA3",
    "CCNA2",
    "GTSE1",
    "CENPF",
    "CKS1B",
    "MELK",
    "CDCA5",
    # Added 4/8 during LPM sub-subclustering
    "MCM4",
    "HIST1H1E",
    "BRCA2",
    "TRIM66",
    "C1orf112",
    "POLD3",
    "KIF23",
    # Added 4/8 part 2
    "CDKAL1",
    "RAD51AP1",
    "RANBP1",
    "CDCA2",
    "CAND2",
    "PCLAF",
    "EIF5A",
    "SRSF2",
    "GINS1",
    "GINS2",
    # Added 4/9 Neuron subclustering
    "ncAPG2",
    "CCND1",
    "ATAD2",
    "UBE2T",
    "CENPK",
    "LAPTM4B",
    "CCND2",
    "MED13",
    "ZNF37A",
    "PTPN14",
    # 4/10 Neural Crest
    "ncAPG",
    "TTK",
    "KIF4A",
    "KIF18A",
    "CDKN3",
    "CEP70",
    "BRIP1",
    "SPC25",
    "KIFC1",
    "NSD2",
    "BUB1",
    "BUB1B",
    "ANP32E",
    "HMGB3",
    "PCNA",
    "CENPP",
    # 4/10 Roof plate
    "CCT5",
    "KIF15",
    "CSRP2",
    "ORC6",
    "HMGB2",
]

In [ ]:
# rp clusters
adata_morph_rp = adata_morph_raw.copy()

# Filter for cells more than 5k reads
rp_minimum_reads = 5000
sc.pp.filter_cells(adata_morph_rp, min_counts=rp_minimum_reads)

# Downsample such that each cell has 10k reads
sc.pp.downsample_counts(
    adata_morph_rp, counts_per_cell=rp_minimum_reads, replace=True, random_state=0
)


adata_morph_rp.obs["leiden_morph"] = adata_morph_with_clusters.obs["leiden_morph"].reindex(
    adata_morph_rp.obs_names
)
missing_cluster_labels = int(adata_morph_rp.obs["leiden_morph"].isna().sum())
if missing_cluster_labels:
    print(
        f"Dropping {missing_cluster_labels} cells not present in adata_morph_with_clusters "
        "before coarse-cluster subsetting."
    )
    adata_morph_rp = adata_morph_rp[adata_morph_rp.obs["leiden_morph"].notna()].copy()

In [ ]:
rp_clusters = ["Roof Plate"]

# Select cells that are in rp clusters
adata_morph_rp = adata_morph_rp[
    (adata_morph_rp.obs.leiden_morph.isin(rp_clusters).values)
]

# Log-transform read counts
sc.pp.log1p(adata_morph_rp)

# For SMD analysis:
# Filter for log-transformed genes with mean value greater than 0.05 and standard deviation greater than 0.05
adata_morph_rp_filtered = filter_genes(
    adata_morph_rp, mean_cutoff=0.05, std_cutoff=0.05
)
# Normalize each gene to have unit variarpe in expression
adata_morph_rp_filtered = normalize_unit_variance(adata_morph_rp_filtered)

In [ ]:
# Filter for coefficient of variation (std deviation / mean) > 1
# Sirpe we have normalized to unit variarpe this is just a mean cutoff < 1
counts = adata_morph_rp_filtered.X
cv_mask = np.mean(counts, axis=0).transpose() < 1
adata_morph_rp_filtered = adata_morph_rp_filtered[:, cv_mask]

In [ ]:
# # Save pre-processed expression data to .npz file, for input to SMD on cluster
rpnc_stage_path = RESULTS_DIR / "intermediates" / "06_rp_nc_subclustering"
rpnc_stage_path.mkdir(parents=True, exist_ok=True)
legacy_rp_input_path = rpnc_stage_path / "morph_rp4.03.npz"
canonical_rp_input_path = rpnc_stage_path / "trunk_morph_rp__smd_input.npz"
for _path in [legacy_rp_input_path, canonical_rp_input_path]:
    scipy.sparse.save_npz(_path, adata_morph_rp_filtered.X)
recommended_n_sub_rp = int(round(0.8 * len(adata_morph_rp_filtered)))
save_json(
    {
        "stage": "06_rp_nc_subclustering",
        "subset_name": "roof_plate",
        "subset_clusters": rp_clusters,
        "minimum_reads": rp_minimum_reads,
        "legacy_smd_input_file": legacy_rp_input_path.name,
        "canonical_smd_input_file": canonical_rp_input_path.name,
        "n_cells_after_preprocessing": int(adata_morph_rp_filtered.n_obs),
        "n_genes_after_preprocessing": int(adata_morph_rp_filtered.n_vars),
        "recommended_n_sub": recommended_n_sub_rp,
        "status": "ready_for_legacy_compatible_smd_mapping",
    },
    rpnc_stage_path / "pre_smd_input_meta_rp.json",
)
print(
    f"{len(adata_morph_rp_filtered)} RP cells after pre-processing\n"
    f"{adata_morph_rp_filtered.shape[1]} genes after preprocessing\n"
    f"Recommended n_sub ~= {recommended_n_sub_rp}"
)


In [ ]:
# For downstream analysis after SMD:
# Only filter genes for non-zero mean and standard deviation:
adata_morph_rp_nofilter = adata_morph_rp.copy()
adata_morph_rp = filter_genes(adata_morph_rp, mean_cutoff=0, std_cutoff=0)
adata_morph_rp = normalize_unit_variance(adata_morph_rp)

### Roof Plate post-SMD processing

In [ ]:
# Load z-score file from morph_rp SMD output
legacy_rp_scores = build_legacy_smd_score_table(
    legacy_adata_path=SCRNASEQ_INPUT_ROOT
    / "legacy/results/intermediates/02_trunk_main/adata_morph_with_clusters.h5ad",
    legacy_cluster_col="leiden_morph",
    legacy_clusters=rp_clusters,
    zscore_path=PROJECT_ROOT / "z_morph_rp4.03_000.npy",
    score_col="z_score_morph_rp",
    raw_adata=adata_morph_raw,
    minimum_reads=rp_minimum_reads,
    mean_cutoff=0.05,
    std_cutoff=0.05,
    cv_mean_cutoff=1.0,
    random_state=0,
)
adata_morph_rp_filtered, rp_score_map_diag = map_legacy_smd_scores_to_adata(
    adata_morph_rp_filtered,
    legacy_rp_scores,
    score_col="z_score_morph_rp",
)
z_scores_morph_rp = adata_morph_rp_filtered.var["z_score_morph_rp"].to_numpy()
print(
    "Mapped legacy morph_rp z-scores to current filtered genes: "
    + f"{rp_score_map_diag['n_matched']} matched, "
    + f"{rp_score_map_diag['n_missing_in_current']} missing in current data, "
    + f"{rp_score_map_diag['n_legacy_only']} legacy-only genes ignored."
)

# Plot top morph_rp SMD z-scores
smd_cutoff = 2

plt.figure(figsize=(12, 4))
plt.hlines(smd_cutoff, -1, 2000, "r")  # z_score cutoff for genes: >= 2
plt.plot(sorted(z_scores_morph_rp)[::-1], "k.", markersize=5)
plt.yscale("log")
plt.ylim(0.01, 1.5 * z_scores_morph_rp.max())
plt.xlim(-1, 1000)
plt.ylabel("morph_rp z-score")
print(
    "\n"
    + str((z_scores_morph_rp > smd_cutoff).sum())
    + " morph_rp genes with z_score >= cutoff"
)
plt.show()

In [ ]:
# Filter for top morph_rp genes by z-score
adata_morph_rp_filtered_ = adata_morph_rp_filtered.copy()
adata_morph_rp_ = adata_morph_rp.copy()

# Select genes with z-score > cutoff
SMDmorph_rpgenes = list(
    adata_morph_rp_filtered_.var_names[
        adata_morph_rp_filtered_.var.z_score_morph_rp > smd_cutoff
    ]
)

manual_addgenes = []
manual_removegenes = []

# Resolve selected gene symbols to gene IDs before matrix subsetting.
SMDmorph_rpgenes = resolve_symbols(adata_morph_rp_, SMDmorph_rpgenes, strict=False, allow_missing=True)
# Resolve cell-cycle symbols to gene IDs before dropping them from clustering features.
cellcycle_gene_ids = set(resolve_symbols(adata_morph_rp_, genes_cellcycle, strict=False, allow_missing=True))
SMDmorph_rpgenes = [g for g in SMDmorph_rpgenes if g not in cellcycle_gene_ids]

adata_morph_rp_.var["z_score_morph_rp"] = None
for g in adata_morph_rp_filtered_.var_names:
    adata_morph_rp_.var.loc[g, "z_score_morph_rp"] = (
        adata_morph_rp_filtered_.var.loc[g, "z_score_morph_rp"]
    )
# Subset columns by resolved gene IDs to keep symbol mapping deterministic.
adata_morph_rp_SMD = adata_morph_rp_[:, resolve_symbols(adata_morph_rp_, SMDmorph_rpgenes, strict=False, allow_missing=True)]

print(
    str(len(adata_morph_rp_SMD.var))
    + " genes selected with morph_rp z > cutoff after cell-cycle removal"
)

In [ ]:
corr_morph_rp_gg = np.corrcoef(adata_morph_rp_SMD.X.todense().T)

cluster_gg_morph_rp = sns.clustermap(
    corr_morph_rp_gg,
    method="ward",
    metric="euclidean",
    figsize=(20, 20),
    cmap=batlow,
    vmin=-0.2,
    vmax=0.8,
    yticklabels=var_names_to_symbols(adata_morph_rp_SMD),
    xticklabels=var_names_to_symbols(adata_morph_rp_SMD),
)

In [ ]:
# Keep the 02-style simplified RP panel: z > cutoff only, with cell-cycle removal.
print("Skipping correlation-based gene pruning and manual gene overrides for roof plate; using the 02-style simplified panel.")


In [ ]:
corr_morph_rp_gg = gene_corrcoef_sparse_safe(adata_morph_rp_SMD.X)

cluster_gg_morph_rp = sns.clustermap(
    corr_morph_rp_gg,
    method="ward",
    metric="euclidean",
    figsize=(20, 20),
    cmap=batlow,
    vmin=-0.2,
    vmax=0.8,
    yticklabels=var_names_to_symbols(adata_morph_rp_SMD),
    xticklabels=var_names_to_symbols(adata_morph_rp_SMD),
)

#### Supp. Data 1 - SMD Roof Plate Z-scores

In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 1, "Gene markers and differential expression supporting trunk morph cell type annotations"
# Sheet 2, "SMD Z-scores"
# Columns G-H "Roof Plate Subclustering"
try:
    adata_morph_rp_SMD.var.z_score_morph_rp.sort_values(ascending=False)
except Exception as exc:
    print(f"Skipping RP clipboard export: {exc}")


### Roof Plate Clustering in SMD gene space

In [ ]:
# Cluster roof plate data independently

# Create a copy and normalize to uniform reads among morph cells
adata_morph_rp_SMD_ = adata_morph_rp_SMD.copy()


sc.pp.normalize_total(adata_morph_rp_SMD_)
try:
    sc.pp.neighbors(adata_morph_rp_SMD_, use_rep="X")
except:
    sc.pp.neighbors(adata_morph_rp_SMD_, use_rep="X")
sc.tl.umap(adata_morph_rp_SMD_, random_state=0)

# Leiden clustering
sc.tl.leiden(
    adata_morph_rp_SMD_,
    resolution=0.62,
    random_state=0,
    key_added="leiden_morph_rp",
    flavor="igraph",
    n_iterations=-1,
)


celltypes_morph_rp = {
    "2": "Dorsal Neuron Progenitors",
    # GRIN2A: subunit of NDMA glutamate receptor involved in synaptic plasticity and learning
    # GRIK1 glutamate receptor, ionotropic
    # SLC5A7: choline transporter crucial for ACh synthesis
    # LYPD1: neurotransmitter receptor-binding proteins
    # GRIA4: subunit of AMPA glutamate receptor
    # ID1 ID4 high
    "1": "Roof Plate Neuroepithelium",
    "0": "Neural Crest Progenitors",
    # NAV3, SOX9, low TFAP2A/B
}

adata_morph_rp_SMD_.obs["leiden_morph_rp"] = adata_morph_rp_SMD_.obs[
    "leiden_morph_rp"
].cat.rename_categories(celltypes_morph_rp)

# Copy cluster IDs to adata with all genes
adata_morph_rp_ = adata_morph_rp.copy()
adata_morph_rp_.obs["leiden_morph_rp"] = adata_morph_rp_SMD_.obs[
    "leiden_morph_rp"
]

adata_morph_rp_SMD_leiden = sc.get.aggregate(
    adata_morph_rp_SMD_,
    by="leiden_morph_rp",
    func=["count_nonzero", "mean", "sum", "var"],
    axis="obs",
)

corr_clcl_morph_rp = np.corrcoef(adata_morph_rp_SMD_leiden.layers["mean"])

with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": (300)}):
    cluster_clcl_morph_rp = sns.clustermap(
        corr_clcl_morph_rp,
        method="ward",
        metric="euclidean",
        figsize=(10, 10),
        cmap=batlow,
        # vmin=-0.2,
        # vmax=1,
        yticklabels=adata_morph_rp_SMD_leiden.obs.leiden_morph_rp.cat.categories,
        xticklabels=adata_morph_rp_SMD_leiden.obs.leiden_morph_rp.cat.categories,
    )
    cluster_clcl_morph_rp.fig.suptitle(
        "Cluster-cluster mean gene correlation among all expressed genes"
    )
    # plt.savefig("figures/cluster-cluster_correlation.pdf")

adata_morph_rp_.obs.leiden_morph_rp = (
    adata_morph_rp_.obs.leiden_morph_rp.cat.reorder_categories(
        adata_morph_rp_.obs.leiden_morph_rp.cat.categories[
            cluster_clcl_morph_rp.dendrogram_col.reordered_ind
        ]
    )
)

adata_morph_rp_SMD_.obs.leiden_morph_rp = (
    adata_morph_rp_SMD_.obs.leiden_morph_rp.cat.reorder_categories(
        adata_morph_rp_SMD_.obs.leiden_morph_rp.cat.categories[
            cluster_clcl_morph_rp.dendrogram_col.reordered_ind
        ]
    )
)

adata_morph_rp_SMD_ = adata_morph_rp_SMD_[
    adata_morph_rp_SMD_.obs.sort_values("leiden_morph_rp").index, :
]
adata_morph_rp_ = adata_morph_rp_[
    adata_morph_rp_.obs.sort_values("leiden_morph_rp").index, :
]

In [ ]:
celltypeorder = [
    "Dorsal Neuron Progenitors",
    "Roof Plate Neuroepithelium",
    "Neural Crest Progenitors",
]
rp_series = adata_morph_rp_SMD_.obs["leiden_morph_rp"].cat.remove_unused_categories()
observed_celltypes = list(rp_series.cat.categories)
missing_expected = [ct for ct in celltypeorder if ct not in observed_celltypes]
unexpected_celltypes = [ct for ct in observed_celltypes if ct not in celltypeorder]
if missing_expected:
    print("RP categories missing from current partition:", missing_expected)
if unexpected_celltypes:
    print("Appending unexpected RP categories at end of ordering:", unexpected_celltypes)
ordered_celltypes = [ct for ct in celltypeorder if ct in observed_celltypes] + unexpected_celltypes
adata_morph_rp_SMD_.obs["leiden_morph_rp"] = rp_series.cat.reorder_categories(
    ordered_celltypes
)
adata_morph_rp_SMD_ = adata_morph_rp_SMD_[
    adata_morph_rp_SMD_.obs.leiden_morph_rp.sort_values().index, :
]

with plt.rc_context({"figure.dpi": (300)}):
    linkage_genes = scipy.cluster.hierarchy.linkage(
        adata_morph_rp_SMD_.X.todense().T,
        method="complete",
        metric="jensenshannon",
        optimal_ordering=True,
    )

    for t_dist in [
        0.7
    ]:  # np.linspace(0.4, 1, 61):  # t_dist 50 splits into one gene per cluster
        print(t_dist)
        adata_morph_rp_SMD_leiden.X = adata_morph_rp_SMD_leiden.layers["mean"]

        adata_morph_rp_SMD_.var["fcluster"] = scipy.cluster.hierarchy.fcluster(
            linkage_genes, t=t_dist, criterion="distance"
        )

        adata_morph_rp_SMD_fcluster = sc.get.aggregate(
            adata_morph_rp_SMD_,
            by="fcluster",
            func=["count_nonzero", "mean", "sum", "var"],
            axis="var",
        )

        adata_morph_rp_SMD_fcluster.X = adata_morph_rp_SMD_fcluster.layers["mean"]

        # Calculate top 100 DEGs per cluster, among all genes, by multinomial logistic regression
        sc.tl.rank_genes_groups(
            adata_morph_rp_SMD_fcluster,
            groupby="leiden_morph_rp",
            method="wilcoxon",
            rankby_abs=False,
            max_iter=1000,
            multi_class="multinomial",
        )
        DEGs_morph_fcluster = pd.DataFrame(
            adata_morph_rp_SMD_fcluster.uns["rank_genes_groups"]["names"]
        )
        DEGscores_morph_fcluster = pd.DataFrame(
            adata_morph_rp_SMD_fcluster.uns["rank_genes_groups"]["scores"]
        )

        for i in adata_morph_rp_SMD_fcluster.var.fcluster.values:
            adata_morph_rp_SMD_fcluster.var.loc[str(i), "leiden"] = (
                DEGscores_morph_fcluster[DEGs_morph_fcluster == str(i)]
                .max(axis=0)
                .idxmax()
            )
            adata_morph_rp_SMD_fcluster.var.loc[str(i), "score"] = (
                DEGscores_morph_fcluster[DEGs_morph_fcluster == str(i)]
                .max(axis=0)
                .max()
            )

        adata_morph_rp_SMD_fcluster.var["leiden"] = (
            adata_morph_rp_SMD_fcluster.var["leiden"].astype(
                pd.CategoricalDtype(categories=celltypeorder, ordered=True)
            )
        )

        print(
            [
                ct
                for ct in celltypeorder
                if ct
                not in set(list(adata_morph_rp_SMD_fcluster.var["leiden"].values))
            ]
        )
        # assert(set(adata_morph_SMD_fcluster.var["leiden"].values) == set(celltypeorder))

        adata_morph_rp_SMD_fcluster = adata_morph_rp_SMD_fcluster[
            :,
            adata_morph_rp_SMD_fcluster.var.sort_values(
                ["leiden", "score"], ascending=[True, False]
            ).index,
        ]

        clusterorder = list(adata_morph_rp_SMD_fcluster.var.index.astype("int32"))

        adata_morph_rp_SMD_.var.fcluster = (
            adata_morph_rp_SMD_.var.fcluster.astype(
                pd.CategoricalDtype(categories=clusterorder, ordered=True)
            )
        )

        adata_morph_rp_SMD_2 = adata_morph_rp_SMD_[
            :, adata_morph_rp_SMD_.var.fcluster.sort_values().index
        ]
        print(len(set(adata_morph_rp_SMD_2.var.fcluster)))
        sc.pl.heatmap(
            adata_morph_rp_SMD_2,
            var_names=adata_morph_rp_SMD_2.var_names,
            groupby="leiden_morph_rp",
            swap_axes=True,
            show_gene_labels=True,
            figsize=(10, 10),
            vmin=0,
            vmax=5,
            cmap=batlow,
        )

### Merge Roof Plate and Neural Crest data

In [ ]:
genes_cellcycle = [
    "MKI67",
    "CENPE",
    "SGO2",
    "KIF14",
    "PIF1",
    "NDC80",
    "CDCA8",
    "PLK1",
    "AURKA",
    "UBE2C",
    "ASPM",
    "TOP2A",
    "TPX2",
    "NUSAP1",
    "CDC20",
    "CKS2",
    "KPNA2",
    "TUBB4B",
    "DLGAP5",
    "BIRC5",
    "HMMR",
    "CCNB1",
    "ARL6IP1",
    "PTTG1",
    "UBE2S",
    "DUT",
    "HELLS",
    "CLSPN",
    "RRM2",
    "PCLAF",
    "TYMS",
    "KIF18B",
    "SMC4",
    "HIST1H4C",
    "DIAPH3",
    "RFC3",
    "MIR924HG",
    "MIS18BP1",
    "TUBA1C",
    "CDK1",
    "CENPA",
    # extra genes added after mesoderm subclustering, were not already in SMD gene list
    "KIF11",
    "ECT2",
    "KNL1",
    "NEK2",
    "CEP55",
    "PSRC1",
    "CDCA3",
    "CCNA2",
    "GTSE1",
    "CENPF",
    "CKS1B",
    "MELK",
    "CDCA5",
    # Added 4/8 during LPM sub-subclustering
    "MCM4",
    "HIST1H1E",
    "BRCA2",
    "TRIM66",
    "C1orf112",
    "POLD3",
    "KIF23",
    # Added 4/8 part 2
    "CDKAL1",
    "RAD51AP1",
    "RANBP1",
    "CDCA2",
    "CAND2",
    "PCLAF",
    "EIF5A",
    "SRSF2",
    "GINS1",
    "GINS2",
    # Added 4/9 Neuron subclustering
    "ncAPG2",
    "CCND1",
    "ATAD2",
    "UBE2T",
    "CENPK",
    "LAPTM4B",
    "CCND2",
    "MED13",
    "ZNF37A",
    "PTPN14",
    # 4/10 Neural Crest
    "ncAPG",
    "TTK",
    "KIF4A",
    "KIF18A",
    "CDKN3",
    "CEP70",
    "BRIP1",
    "SPC25",
    "KIFC1",
    "NSD2",
    "BUB1",
    "BUB1B",
    "ANP32E",
    "HMGB3",
    "PCNA",
    "CENPP",
    # 4/10 Roof plate
    "CCT5",
    "KIF15",
    "CSRP2",
    "ORC6",
    "HMGB2",
]

In [ ]:
# rpnc clusters
adata_morph_rpnc = adata_morph_raw.copy()

# Filter for cells more than 5k reads
rpnc_minimum_reads = 5000
sc.pp.filter_cells(adata_morph_rpnc, min_counts=rpnc_minimum_reads)

# Downsample such that each cell has 10k reads
sc.pp.downsample_counts(
    adata_morph_rpnc, counts_per_cell=rpnc_minimum_reads, replace=True, random_state=0
)


adata_morph_rpnc.obs["leiden_morph"] = adata_morph_with_clusters.obs["leiden_morph"].reindex(
    adata_morph_rpnc.obs_names
)
missing_cluster_labels = int(adata_morph_rpnc.obs["leiden_morph"].isna().sum())
if missing_cluster_labels:
    print(
        f"Dropping {missing_cluster_labels} cells not present in adata_morph_with_clusters "
        "before coarse-cluster subsetting."
    )
    adata_morph_rpnc = adata_morph_rpnc[adata_morph_rpnc.obs["leiden_morph"].notna()].copy()

In [ ]:
rpnc_clusters = ["Roof Plate", "Neural Crest"]

# Select cells that are in rpnc clusters
adata_morph_rpnc = adata_morph_rpnc[
    (adata_morph_rpnc.obs.leiden_morph.isin(rpnc_clusters).values)
]

# Log-transform read counts
sc.pp.log1p(adata_morph_rpnc)

### Combined RP / NC post-SMD processing

In [ ]:
# Prepare combined RP / NC SMD input before the historical separate downstream analyses.
adata_morph_rpnc_filtered = filter_genes(
    adata_morph_rpnc, mean_cutoff=0.05, std_cutoff=0.05
)
adata_morph_rpnc_filtered = normalize_unit_variance(adata_morph_rpnc_filtered)

rpnc_stage_path = RESULTS_DIR / "intermediates" / "06_rp_nc_subclustering"
rpnc_stage_path.mkdir(parents=True, exist_ok=True)
canonical_rpnc_input_path = rpnc_stage_path / "trunk_morph_rp_nc__smd_input.npz"
scipy.sparse.save_npz(canonical_rpnc_input_path, adata_morph_rpnc_filtered.X)
recommended_n_sub_rpnc = int(round(0.8 * len(adata_morph_rpnc_filtered)))
save_json(
    {
        "stage": "06_rp_nc_subclustering",
        "subset_name": "roof_plate_neural_crest_combined",
        "subset_clusters": rpnc_clusters,
        "minimum_reads": rpnc_minimum_reads,
        "canonical_smd_input_file": canonical_rpnc_input_path.name,
        "n_cells_after_preprocessing": int(adata_morph_rpnc_filtered.n_obs),
        "n_genes_after_preprocessing": int(adata_morph_rpnc_filtered.n_vars),
        "recommended_n_sub": recommended_n_sub_rpnc,
        "source_counts": {str(k): int(v) for k, v in adata_morph_rpnc.obs["source"].astype(str).value_counts().items()},
        "coarse_label_counts": {str(k): int(v) for k, v in adata_morph_rpnc.obs["leiden_morph"].astype(str).value_counts().items()},
        "status": "ready_for_smd",
    },
    rpnc_stage_path / "pre_smd_input_meta_rpnc.json",
)
print(
    f"{len(adata_morph_rpnc_filtered)} combined RP/NC cells after pre-processing\n"
    f"{adata_morph_rpnc_filtered.shape[1]} genes after preprocessing\n"
    f"Recommended n_sub ~= {recommended_n_sub_rpnc}"
)


In [ ]:
adata_morph_rpnc = filter_genes(adata_morph_rpnc, mean_cutoff=0, std_cutoff=0)
adata_morph_rpnc = normalize_unit_variance(adata_morph_rpnc)

In [ ]:
# Load modern combined RP / NC SMD runs from local trunk_main_dev results
import re

rpnc_smd_run_order = ["001"]
rpnc_smd_run_dirs = {
    "001": PROJECT_ROOT / "trunk_main_dev" / "results" / "smd_runs" / "trunk_morph_rpnc__run001",
}
rpnc_smd_runs = {
    run_id: load_npy(
        rpnc_smd_run_dirs[run_id] / f"z_trunk_morph_rp_nc__smd_input_{run_id}.npy"
    )
    for run_id in rpnc_smd_run_order
}
selected_rpnc_smd_run = "001"
rpnc_smd_run_name = f"trunk_morph_rpnc__run{selected_rpnc_smd_run}"
rpnc_smd_zscore_name = f"z_trunk_morph_rp_nc__smd_input_{selected_rpnc_smd_run}.npy"
z_scores_morph_rpnc = rpnc_smd_runs[selected_rpnc_smd_run]

if len(z_scores_morph_rpnc) != adata_morph_rpnc_filtered.n_vars:
    raise ValueError(
        f"Combined RP/NC SMD run {selected_rpnc_smd_run} has {len(z_scores_morph_rpnc)} genes, "
        f"expected {adata_morph_rpnc_filtered.n_vars}."
    )

rpnc_run_config_text = (rpnc_smd_run_dirs[selected_rpnc_smd_run] / "ray_SMD.py").read_text()
match_n_sub = re.search(r"n_sub\s*=\s*(\d+)", rpnc_run_config_text)
match_trials = re.search(r"trials\s*=\s*(\d+)", rpnc_run_config_text)
rpnc_smd_run_n_sub = int(match_n_sub.group(1)) if match_n_sub else None
rpnc_smd_run_trials = int(match_trials.group(1)) if match_trials else None

rpnc_smd_run_summary = pd.DataFrame(
    [
        {
            "run_id": run_id,
            "n_sub": int(rpnc_smd_run_n_sub) if run_id == selected_rpnc_smd_run and rpnc_smd_run_n_sub is not None else np.nan,
            "trials": int(rpnc_smd_run_trials) if run_id == selected_rpnc_smd_run and rpnc_smd_run_trials is not None else np.nan,
            "n_genes_z_gt_2": int((rpnc_smd_runs[run_id] > 2).sum()),
            "n_genes_z_gt_10": int((rpnc_smd_runs[run_id] > 10).sum()),
            "max_z": float(rpnc_smd_runs[run_id].max()),
        }
        for run_id in rpnc_smd_run_order
    ]
)
display(rpnc_smd_run_summary)

rpnc_smd_cutoff = 2.0

with plt.rc_context({"figure.dpi": 110}):
    plt.figure(figsize=(7, 3))
    plt.hlines(rpnc_smd_cutoff, -1, 2000, "r")
    plt.plot(sorted(z_scores_morph_rpnc)[::-1], "k.", markersize=5)
    plt.yscale("log")
    plt.ylim(0.01, 1.5 * z_scores_morph_rpnc.max())
    plt.xlim(-1, 300)
    plt.ylabel("combined RP/NC SMD z-score")
    plt.text(
        150,
        100,
        f"{(z_scores_morph_rpnc > rpnc_smd_cutoff).sum()} combined RP/NC genes with z_score > {rpnc_smd_cutoff:g}",
    )
    plt.show()

print(
    f"Using combined RP/NC SMD run {selected_rpnc_smd_run} "
    + (f"(n_sub={rpnc_smd_run_n_sub}) " if rpnc_smd_run_n_sub is not None else "")
    + "for parity-inspired combined RP/NC precluster substrate export."
)



In [ ]:
# Build the modern combined RP / NC selected-gene panel using the reconstructed parity 06 union.
adata_morph_rpnc_filtered_ = adata_morph_rpnc_filtered.copy()
adata_morph_rpnc_ = adata_morph_rpnc.copy()

adata_morph_rpnc_filtered_.var["z_score_morph_rpnc"] = z_scores_morph_rpnc
adata_morph_rpnc_filtered_.var["log1p_z_score_morph_rpnc"] = np.log1p(
    np.clip(z_scores_morph_rpnc, 0, None)
)

parity_06_union_gene_ids = ['ENSG00000188290', 'ENSG00000189337', 'ENSG00000160752', 'ENSG00000143320', 'ENSG00000162761', 'ENSG00000143344', 'ENSG00000143476', 'ENSG00000138101', 'ENSG00000183023', 'ENSG00000115665', 'ENSG00000150551', 'ENSG00000182263', 'ENSG00000286797', 'ENSG00000163931', 'ENSG00000185008', 'ENSG00000169855', 'ENSG00000239268', 'ENSG00000169744', 'ENSG00000138640', 'ENSG00000164109', 'ENSG00000198589', 'ENSG00000145555', 'ENSG00000172201', 'ENSG00000135333', 'ENSG00000146263', 'ENSG00000120437', 'ENSG00000174469', 'ENSG00000104549', 'ENSG00000154529', 'ENSG00000078725', 'ENSG00000167081', 'ENSG00000151474', 'ENSG00000078114', 'ENSG00000174721', 'ENSG00000149256', 'ENSG00000152578', 'ENSG00000064309', 'ENSG00000067798', 'ENSG00000255794', 'ENSG00000102531', 'ENSG00000183098', 'ENSG00000175198', 'ENSG00000166165', 'ENSG00000248905', 'ENSG00000103275', 'ENSG00000078328', 'ENSG00000109084', 'ENSG00000239672', 'ENSG00000141376', 'ENSG00000125398', 'ENSG00000141552', 'ENSG00000267313', 'ENSG00000267586', 'ENSG00000074657', 'ENSG00000167670', 'ENSG00000271774', 'ENSG00000171189', 'ENSG00000133424', 'ENSG00000101849', 'ENSG00000126733', 'ENSG00000123560', 'ENSG00000129682', 'ENSG00000183454', 'ENSG00000137203', 'ENSG00000019549', 'ENSG00000181449', 'ENSG00000145423', 'ENSG00000082701', 'ENSG00000118495', 'ENSG00000120093', 'ENSG00000197921', 'ENSG00000173406', 'ENSG00000132854', 'ENSG00000117069', 'ENSG00000155368', 'ENSG00000121989', 'ENSG00000116117', 'ENSG00000231304', 'ENSG00000144857', 'ENSG00000114805', 'ENSG00000109339', 'ENSG00000182168', 'ENSG00000151617', 'ENSG00000113389', 'ENSG00000175471', 'ENSG00000146242', 'ENSG00000146374', 'ENSG00000152818', 'ENSG00000131016', 'ENSG00000153721', 'ENSG00000122585', 'ENSG00000135205', 'ENSG00000286214', 'ENSG00000253554', 'ENSG00000165084', 'ENSG00000091656', 'ENSG00000106829', 'ENSG00000148219', 'ENSG00000169126', 'ENSG00000099250', 'ENSG00000066468', 'ENSG00000183715', 'ENSG00000111728', 'ENSG00000139219', 'ENSG00000166426', 'ENSG00000258947', 'ENSG00000158270', 'ENSG00000064655', 'ENSG00000101134', 'ENSG00000107105', 'ENSG00000100146', 'ENSG00000118257']
parity_06_union_gene_symbols = ['HES4', 'KAZN', 'FDPS', 'CRABP2', 'LMX1A', 'RGL1', 'DTL', 'DTNB', 'SLC8A1', 'SLC5A7', 'LYPD1', 'FIGN', 'AC009315.1', 'TKT', 'ROBO2', 'ROBO1', 'AC092691.1', 'LDB2', 'FAM13A', 'MAD2L1', 'LRBA', 'MYO10', 'ID4', 'EPHA7', 'MMS22L', 'ACAT2', 'CNTNAP2', 'SQLE', 'CNTNAP3B', 'BRINP1', 'PBX3', 'FRMD4A', 'NEBL', 'FGFBP3', 'TENM4', 'GRIA4', 'CDON', 'NAV3', 'RMST', 'FNDC3A', 'GPC6', 'PCCA', 'CKB', 'FMN1', 'UBE2I', 'RBFOX1', 'TMEM97', 'NME1', 'BCAS3', 'SOX9', 'ANAPC11', 'AC021504.1', 'LINC00907', 'ZNF532', 'CHAF1A', 'AL109930.1', 'GRIK1', 'LARGE1', 'TBL1X', 'DACH2', 'PLP1', 'FGF13', 'GRIN2A', 'TFAP2A', 'SNAI2', 'SOX2', 'SFRP2', 'GSK3B', 'PLAGL1', 'HOXB3', 'HES5', 'DAB1', 'KANK4', 'ST6GALNAC5', 'DBI', 'ACVR2A', 'PARD3B', 'SGO1-AS1', 'BOC', 'PLCH1', 'MAPK10', 'UNC5C', 'EDNRA', 'NPR3', 'MCTP1', 'TPBG', 'RSPO3', 'UTRN', 'AKAP12', 'CNKSR3', 'NPY', 'CCDC146', 'AUXG01000058.1', 'LINC01414', 'C8orf34', 'ZFHX4', 'TLE4', 'ASTN2', 'ARMC4', 'NRP1', 'FGFR2', 'OPCML', 'ST8SIA1', 'COL2A1', 'CRABP1', 'TUBB3', 'COLEC12', 'EYA2', 'DOK5', 'ELAVL2', 'SOX10', 'NRP2']

missing_parity_union_gene_ids = [
    gene_id for gene_id in parity_06_union_gene_ids if gene_id not in adata_morph_rpnc_.var_names
]
if missing_parity_union_gene_ids:
    raise ValueError(
        "Parity 06 union gene IDs missing from combined RP/NC object: "
        + ", ".join(missing_parity_union_gene_ids[:10])
        + (" ..." if len(missing_parity_union_gene_ids) > 10 else "")
    )

selected_rpnc_gene_ids = parity_06_union_gene_ids.copy()
rpnc_manual_addgenes = []
rpnc_manual_removegenes = []
rpnc_manual_add_ids = []
rpnc_manual_remove_ids = set()
cellcycle_gene_ids = set(
    resolve_symbols(adata_morph_rpnc_, genes_cellcycle, strict=False, allow_missing=True)
)

adata_morph_rpnc_.var["z_score_morph_rpnc"] = np.nan
adata_morph_rpnc_.var.loc[
    adata_morph_rpnc_filtered_.var_names, "z_score_morph_rpnc"
] = adata_morph_rpnc_filtered_.var["z_score_morph_rpnc"].to_numpy()
adata_morph_rpnc_.var["log1p_z_score_morph_rpnc"] = np.log1p(
    np.clip(
        pd.to_numeric(adata_morph_rpnc_.var["z_score_morph_rpnc"], errors="coerce")
        .fillna(0)
        .to_numpy(),
        0,
        None,
    )
)

adata_morph_rpnc_SMD = adata_morph_rpnc_[:, selected_rpnc_gene_ids].copy()

selected_rpnc_scores = adata_morph_rpnc_filtered_.var.reindex(selected_rpnc_gene_ids)["z_score_morph_rpnc"]
print(
    f"Using reconstructed parity 06 RP/NC union panel with {len(selected_rpnc_gene_ids)} genes on the modern combined object."
)
print(
    f"Parity union support in modern combined run: z > 2 for {int((selected_rpnc_scores > 2).sum())} genes; "
    + f"z > 10 for {int((selected_rpnc_scores > 10).sum())} genes."
)



In [ ]:
# Export the combined RP / NC clustering substrate for leiden_ensemble/06.
adata_morph_rpnc_SMD_ = adata_morph_rpnc_SMD.copy()

sc.pp.normalize_total(adata_morph_rpnc_SMD_)
try:
    sc.pp.neighbors(adata_morph_rpnc_SMD_, use_rep="X")
except Exception:
    sc.pp.neighbors(adata_morph_rpnc_SMD_, use_rep="X")
sc.tl.umap(adata_morph_rpnc_SMD_, random_state=0)

print(
    f"Prepared combined RP/NC selected-gene substrate with {adata_morph_rpnc_SMD_.n_obs} cells and {adata_morph_rpnc_SMD_.n_vars} genes."
)
print(
    "Combined RP/NC clustering and label assignment are intentionally deferred to leiden_ensemble/06."
)


In [ ]:
# Summarize the combined RP / NC SMD gene scores and final selected-gene panel.
rpnc_gene_scores = pd.DataFrame(
    {
        "gene_ids": adata_morph_rpnc_filtered_.var_names.astype(str),
        "gene_symbol": adata_morph_rpnc_filtered_.var["gene_symbol"].astype(str).to_numpy(),
        "gene_symbol_original": adata_morph_rpnc_filtered_.var["gene_symbol_original"].astype(str).to_numpy(),
        "z_score_morph_rpnc": adata_morph_rpnc_filtered_.var["z_score_morph_rpnc"].to_numpy(),
        "log1p_z_score_morph_rpnc": adata_morph_rpnc_filtered_.var["log1p_z_score_morph_rpnc"].to_numpy(),
        "selected_at_z_gt_2": (adata_morph_rpnc_filtered_.var["z_score_morph_rpnc"].to_numpy() > 2),
        "selected_at_z_gt_current": (
            adata_morph_rpnc_filtered_.var["z_score_morph_rpnc"].to_numpy() > rpnc_smd_cutoff
        ),
        "included_in_parity_06_union_panel": adata_morph_rpnc_filtered_.var_names.isin(parity_06_union_gene_ids),
        "z_threshold_current": rpnc_smd_cutoff,
        "smd_run_id": selected_rpnc_smd_run,
        "smd_run_n_sub": rpnc_smd_run_n_sub,
    }
)

selection_gene_ids = list(adata_morph_rpnc_filtered_.var_names.astype(str))
selection_gene_id_set = set(selection_gene_ids)
for extra_ids in [parity_06_union_gene_ids, cellcycle_gene_ids]:
    for gene_id in extra_ids:
        gene_id = str(gene_id)
        if gene_id not in selection_gene_id_set:
            selection_gene_ids.append(gene_id)
            selection_gene_id_set.add(gene_id)

selection_gene_index = pd.Index(selection_gene_ids, dtype="object")
selection_meta = adata_morph_rpnc_.var.reindex(selection_gene_index)
selection_scores = adata_morph_rpnc_filtered_.var.reindex(selection_gene_index)
rpnc_final_gene_selection = pd.DataFrame(
    {
        "gene_ids": selection_gene_index.astype(str),
        "gene_symbol": selection_meta["gene_symbol"].astype("string").to_numpy(),
        "gene_symbol_original": selection_meta["gene_symbol_original"].astype("string").to_numpy(),
        "in_smd_scored_space": selection_gene_index.isin(adata_morph_rpnc_filtered_.var_names),
        "z_score_morph_rpnc": selection_scores["z_score_morph_rpnc"].to_numpy(),
        "log1p_z_score_morph_rpnc": selection_scores["log1p_z_score_morph_rpnc"].to_numpy(),
        "selected_at_z_gt_2": (
            selection_scores["z_score_morph_rpnc"].fillna(float("-inf")).to_numpy() > 2
        ),
        "selected_at_z_gt_current": (
            selection_scores["z_score_morph_rpnc"].fillna(float("-inf")).to_numpy()
            > rpnc_smd_cutoff
        ),
        "included_in_parity_06_union_panel": selection_gene_index.isin(parity_06_union_gene_ids),
        "manually_added_to_final_smd": selection_gene_index.isin(rpnc_manual_add_ids),
        "manually_removed_from_final_smd": selection_gene_index.isin(rpnc_manual_remove_ids),
        "removed_as_cell_cycle": selection_gene_index.isin(cellcycle_gene_ids),
        "included_in_final_smd_current": selection_gene_index.isin(adata_morph_rpnc_SMD.var_names),
        "z_threshold_current": rpnc_smd_cutoff,
        "smd_run_id": selected_rpnc_smd_run,
        "smd_run_n_sub": rpnc_smd_run_n_sub,
    }
)

display(rpnc_gene_scores.sort_values("z_score_morph_rpnc", ascending=False).head(20))
print(
    "Combined RP/NC selected genes in reconstructed parity 06 union panel: "
    + str(int(rpnc_final_gene_selection["included_in_final_smd_current"].sum()))
)
print(
    "Of those, genes also in the modern combined scored space: "
    + str(int((rpnc_final_gene_selection["included_in_final_smd_current"] & rpnc_final_gene_selection["in_smd_scored_space"]).sum()))
)



In [ ]:
# Legacy combined RP/NC heatmap assembly is deprecated here; modern combined clustering is deferred to leiden_ensemble/06.

In [ ]:
# No-op: combined RP/NC figure generation deferred to leiden_ensemble/06.

### Combined RP / NC downstream clustering is deferred to `leiden_ensemble/06`
This notebook now exports the modern combined RP/NC precluster substrates only.

In [ ]:
# No-op: combined RP/NC DEG analysis deferred to leiden_ensemble/06.

#### Supp. Data 1 - Combined RP / NC SMD Z-scores

In [ ]:
# No-op: supplementary DEG export deferred to leiden_ensemble/06.

### Fig. 4d preparation is deferred
Combined RP/NC figure generation should follow the final `leiden_ensemble/06` clustering decision.

In [ ]:
# No-op: RP/NC manuscript heatmap preparation deferred to post-ensemble finalization.

In [ ]:
# No-op: RP/NC key-gene panel figure generation deferred to post-ensemble finalization.

In [ ]:
# No-op: RP/NC key-gene panel figure generation deferred to post-ensemble finalization.

In [ ]:
# No-op: RP/NC key-gene panel figure generation deferred to post-ensemble finalization.

In [ ]:
# No-op: RP/NC key-gene panel figure generation deferred to post-ensemble finalization.

## Save Stage Outputs
Persist legacy split reference objects plus the modern combined RP/NC precluster substrates for downstream `leiden_ensemble/06` work.

In [ ]:
stage_path = stage_dir(RESULTS_DIR, '06_rp_nc_subclustering')
for _adata in [
    adata_morph_nc_,
    adata_morph_nc_SMD_,
    adata_morph_rp_,
    adata_morph_rp_SMD_,
    adata_morph_rpnc_,
    adata_morph_rpnc_filtered,
    adata_morph_rpnc_SMD,
    adata_morph_rpnc_SMD_,
]:
    assert_gene_id_index(_adata)

save_h5ad(adata_morph_nc_, stage_path / 'adata_morph_nc_.h5ad')
save_h5ad(adata_morph_nc_SMD_, stage_path / 'adata_morph_nc_SMD_.h5ad')
save_h5ad(adata_morph_rp_, stage_path / 'adata_morph_rp_.h5ad')
save_h5ad(adata_morph_rp_SMD_, stage_path / 'adata_morph_rp_SMD_.h5ad')
save_h5ad(adata_morph_rpnc_, stage_path / 'adata_morph_rpnc.h5ad')
save_h5ad(adata_morph_rpnc_filtered, stage_path / 'adata_morph_rpnc_pre_smd_.h5ad')
save_h5ad(adata_morph_rpnc_SMD, stage_path / 'adata_morph_rpnc_SMD.h5ad')
save_h5ad(adata_morph_rpnc_SMD_, stage_path / 'adata_morph_rpnc_SMD_.h5ad')

save_pickle(rpnc_gene_scores, stage_path / 'trunk_morph_rpnc_smd_gene_scores.pkl')
rpnc_gene_scores.to_csv(stage_path / 'trunk_morph_rpnc_smd_gene_scores.csv', index=False)
save_pickle(rpnc_final_gene_selection, stage_path / 'trunk_morph_rpnc_final_gene_selection.pkl')
rpnc_final_gene_selection.to_csv(stage_path / 'trunk_morph_rpnc_final_gene_selection.csv', index=False)

save_json(
    {
        "stage": "06_rp_nc_subclustering",
        "status": "precluster_substrate_exported",
        "seed_policy": "all seeds set to 0",
        "legacy_smd_reference_stage": "legacy/results/intermediates/02_trunk_main/adata_morph_with_clusters.h5ad",
        "preferred_fresh_smd_substrate": "combined_rp_nc",
        "rpnc_combined_smd_input_file": "trunk_morph_rp_nc__smd_input.npz",
        "selected_rpnc_smd_run": selected_rpnc_smd_run,
        "selected_rpnc_smd_run_name": rpnc_smd_run_name,
        "selected_rpnc_smd_run_file": rpnc_smd_zscore_name,
        "selected_rpnc_smd_run_n_sub": rpnc_smd_run_n_sub,
        "selected_rpnc_smd_run_trials": rpnc_smd_run_trials,
        "available_rpnc_smd_runs": rpnc_smd_run_order,
        "rpnc_combined_clusters": rpnc_clusters,
        "nc_subset_clusters": nc_clusters,
        "rp_subset_clusters": rp_clusters,
        "smd_cutoff_current": rpnc_smd_cutoff,
        "manual_add_genes": rpnc_manual_addgenes,
        "manual_remove_genes": rpnc_manual_removegenes,
        "gene_selection_rule": "parity_06_union_panel_replayed_on_modern_combined_object",
        "parity_06_union_panel_size": int(len(parity_06_union_gene_ids)),
        "parity_06_union_panel_symbols": parity_06_union_gene_symbols,
        "parity_06_union_panel_missing_in_current_object": int(len(missing_parity_union_gene_ids)),
        "parity_source_notebook": "parity/notebooks/06_rp_nc_subclustering.ipynb",
        "n_cells_input_nc": int(adata_morph_nc_.n_obs),
        "n_cells_input_rp": int(adata_morph_rp_.n_obs),
        "n_cells_combined_rpnc": int(adata_morph_rpnc_.n_obs),
        "n_cells_after_preprocessing_rpnc": int(adata_morph_rpnc_filtered.n_obs),
        "n_genes_after_preprocessing_rpnc": int(adata_morph_rpnc_filtered.n_vars),
        "recommended_n_sub_rpnc": int(round(0.8 * len(adata_morph_rpnc_filtered))),
        "n_smd_genes_final_rpnc": int(adata_morph_rpnc_SMD_.n_vars),
        "n_smd_genes_final_nc_legacy_split": int(adata_morph_nc_SMD_.n_vars),
        "n_smd_genes_final_rp_legacy_split": int(adata_morph_rp_SMD_.n_vars),
        "legacy_split_outputs_retained_for_reference": True,
    },
    stage_path / 'meta.json',
)
print(f'Saved RP/NC intermediates to {stage_path}')



## Adopt Ensemble Labels and Restore RP/NC Visualizations
Pull the current `leiden_ensemble/06` practical labels back into the dev-stage combined RP/NC objects, then regenerate the key inspection figures and DEG tables inline.


In [ ]:
import json
import re

stage_path = stage_dir(RESULTS_DIR, "06_rp_nc_subclustering")
figures_dir = stage_path / "figures"
tables_dir = stage_path / "tables"
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# One Leiden run on the selected-gene graph at resolution 0.75, seed 5.
chosen_rpnc_resolution = 0.75
RPNC_LEIDEN_RANDOM_STATE = 5
sc.tl.leiden(
    adata_morph_rpnc_SMD_,
    resolution=chosen_rpnc_resolution,
    random_state=RPNC_LEIDEN_RANDOM_STATE,
    key_added="leiden_morph_rpnc_raw",
    flavor="igraph",
    n_iterations=-1,
)

cluster_id_to_raw_label = {
    0: "Roof Plate Neuroepithelium [c0]",
    1: "Dorsal Neuron Progenitors [c1]",
    2: "Migratory Cranial Neural Crest [c2]",
    3: "Neural Crest-Derived Immature Neurons [c3]",
}
raw_order = [cluster_id_to_raw_label[cluster_id] for cluster_id in sorted(cluster_id_to_raw_label)]
base_order = [re.sub(r" \[c\d+\]$", "", label) for label in raw_order]
use_base_labels = len(set(base_order)) == len(base_order)
cluster_label_order = base_order if use_base_labels else raw_order
raw_to_adopted = {
    raw_label: (re.sub(r" \[c\d+\]$", "", raw_label) if use_base_labels else raw_label)
    for raw_label in raw_order
}

cluster_ids = adata_morph_rpnc_SMD_.obs["leiden_morph_rpnc_raw"].astype(int).loc[adata_morph_rpnc_.obs_names]
raw_labels = cluster_ids.map(cluster_id_to_raw_label)
if raw_labels.isna().any():
    missing_ids = sorted(cluster_ids[raw_labels.isna()].unique().tolist())
    raise ValueError(f"Unmapped RP/NC representative cluster IDs: {missing_ids}")
base_labels = raw_labels.str.replace(r" \[c\d+\]$", "", regex=True)
adopted_labels = raw_labels.map(raw_to_adopted)

for _adata in [adata_morph_rpnc_, adata_morph_rpnc_filtered, adata_morph_rpnc_SMD, adata_morph_rpnc_SMD_]:
    _adata.obs["leiden_morph_rpnc_raw"] = pd.Categorical(raw_labels.reindex(_adata.obs_names))
    _adata.obs["leiden_morph_rpnc_base"] = pd.Categorical(base_labels.reindex(_adata.obs_names))
    _adata.obs["leiden_morph_rpnc"] = pd.Categorical(
        adopted_labels.reindex(_adata.obs_names), categories=cluster_label_order, ordered=True
    )

rpnc_cluster_labels = pd.DataFrame(
    {
        "cell_id": adata_morph_rpnc_.obs_names.astype(str),
        "cluster_id": cluster_ids.astype(int).to_numpy(),
        "leiden_morph_rpnc_raw": raw_labels.astype(str).to_numpy(),
        "leiden_morph_rpnc_base": base_labels.astype(str).to_numpy(),
        "leiden_morph_rpnc": adopted_labels.astype(str).to_numpy(),
        "source": adata_morph_rpnc_.obs["source"].astype(str).to_numpy(),
    }
)
rpnc_cluster_labels.to_csv(tables_dir / "rpnc_cluster_labels.csv", index=False)

for _adata in [adata_morph_rpnc_, adata_morph_rpnc_filtered, adata_morph_rpnc_SMD, adata_morph_rpnc_SMD_]:
    if hasattr(_adata.obs["leiden_morph_rpnc"], "cat"):
        _adata.obs["leiden_morph_rpnc"] = _adata.obs["leiden_morph_rpnc"].cat.remove_unused_categories()

with plt.rc_context({"figure.dpi": 300}):
    sc.pl.umap(
        adata_morph_rpnc_SMD_,
        color="leiden_morph_rpnc",
        size=180,
        title="Day 6 Morph Combined Roof Plate / Neural Crest Cells",
        show=False,
    )
    plt.savefig(figures_dir / "rpnc_umap_clusters.png", bbox_inches="tight", pad_inches=0)
    plt.savefig(figures_dir / "rpnc_umap_clusters.pdf", bbox_inches="tight", pad_inches=0)
    plt.show()

source_composition = pd.crosstab(
    adata_morph_rpnc_SMD_.obs["leiden_morph_rpnc"],
    adata_morph_rpnc_SMD_.obs["source"],
    normalize="columns",
).T
source_composition.to_csv(tables_dir / "rpnc_source_stackedbar.csv")
with plt.rc_context({"figure.dpi": 300}):
    fig, ax = plt.subplots()
    source_composition.plot(kind="bar", stacked=True, ax=ax)
    ax.legend(title="Cluster", bbox_to_anchor=(1.7, 1.02), loc="upper right")
    ax.set_ylabel("Fraction of cells")
    ax.set_xlabel("")
    plt.savefig(figures_dir / "rpnc_stackedbar_source_composition.png", bbox_inches="tight", pad_inches=0)
    plt.savefig(figures_dir / "rpnc_stackedbar_source_composition.pdf", bbox_inches="tight", pad_inches=0)
    plt.show()

rpnc_cluster_means = sc.get.aggregate(
    adata_morph_rpnc_SMD_,
    by="leiden_morph_rpnc",
    func=["mean"],
)
rpnc_cluster_means.X = rpnc_cluster_means.layers["mean"]
rpnc_corr = np.corrcoef(np.asarray(rpnc_cluster_means.X))
with plt.rc_context({"figure.dpi": 300}):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        rpnc_corr,
        cmap=batlow,
        vmin=-1,
        vmax=1,
        square=True,
        xticklabels=rpnc_cluster_means.obs_names,
        yticklabels=rpnc_cluster_means.obs_names,
        ax=ax,
    )
    ax.set_title("RP/NC Cluster-Cluster Correlation")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.savefig(figures_dir / "rpnc_cluster_cluster_correlation.png", bbox_inches="tight", pad_inches=0)
    plt.savefig(figures_dir / "rpnc_cluster_cluster_correlation.pdf", bbox_inches="tight", pad_inches=0)
    plt.show()

sc.tl.rank_genes_groups(
    adata_morph_rpnc_,
    groupby="leiden_morph_rpnc",
    method="logreg",
    rankby_abs=False,
    max_iter=1000,
    multi_class="multinomial",
    key_added="noneg",
)
DEGs_morph_multi_rpnc = pd.DataFrame(adata_morph_rpnc_.uns["noneg"]["names"][0:100])
DEGscores_morph_multi_rpnc = pd.DataFrame(adata_morph_rpnc_.uns["noneg"]["scores"][0:100])
DEGs_morph_multi_rpnc.to_csv(tables_dir / "rpnc_top100_logreg_deg_names.csv", index=False)
DEGscores_morph_multi_rpnc.to_csv(tables_dir / "rpnc_top100_logreg_deg_scores.csv", index=False)
rpnc_deg_long = pd.concat(
    [
        pd.DataFrame(
            {
                "cluster": cluster,
                "rank": np.arange(1, len(DEGs_morph_multi_rpnc[cluster]) + 1),
                "gene": DEGs_morph_multi_rpnc[cluster].astype(str).to_numpy(),
                "score": DEGscores_morph_multi_rpnc[cluster].to_numpy(),
            }
        )
        for cluster in DEGs_morph_multi_rpnc.columns
    ],
    ignore_index=True,
)
rpnc_deg_long.to_csv(tables_dir / "rpnc_top100_logreg_deg_long.csv", index=False)

adata_morph_rpnc_bead = adata_morph_rpnc_[adata_morph_rpnc_.obs.source == "BMP4 Bead Morph"].copy()
rpnc_order_genes = resolve_symbols(adata_morph_rpnc_bead, ["MSX1", "ZIC1", "ZIC2"] + parity_06_union_gene_symbols, strict=False, allow_missing=True)
rpnc_heatmap_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_rpnc_bead[:, rpnc_order_genes].copy(),
    cluster_key="leiden_morph_rpnc",
    var_names=rpnc_order_genes,
    figsize=(5, 5),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
pd.DataFrame({"cell_id": rpnc_heatmap_cell_order}).to_csv(tables_dir / "rpnc_heatmap_cell_order.csv", index=False)
adata_morph_rpnc_sorted = adata_morph_rpnc_bead[rpnc_heatmap_cell_order, :].copy()

rpnc_genes = [
    "GRIA4", "GRIN2A", "SLC5A7", "DACH2", "BRINP1", "ZIC1", "GRIK1", "LYPD1", "ZIC2", "UNC5C",
    "TPBG", "PLCH1", "MAPK10", "MSX1", "ZFHX4", "LMX1A", "SOX2", "PLAGL1", "MAD2L1", "DTL",
    "CHAF1A", "NAV3", "SOX9", "PLP1", "TFAP2A", "SNAI2", "SOX10", "NPR3", "EDNRA", "KANK4",
    "HES5", "NRP1", "TUBB3", "DOK5", "ELAVL2", "NPY", "MCTP1", "EYA2",
]
rpnc_missing_genes = symbols_missing(adata_morph_rpnc_sorted, rpnc_genes)
if rpnc_missing_genes:
    print(f"Skipping {len(rpnc_missing_genes)} missing RP/NC heatmap genes: {rpnc_missing_genes}")
rpnc_genes = symbols_present(adata_morph_rpnc_sorted, rpnc_genes)
assert rpnc_genes, "No RP/NC heatmap genes resolved in current bead-only object."

rpnc_bold_genes = [
    "GRIA4", "GRIN2A", "SLC5A7", "ZIC1", "GRIK1", "ZIC2", "MSX1", "ZFHX4", "LMX1A", "SOX2",
    "SOX9", "TFAP2A", "SNAI2", "SOX10", "NPR3", "EDNRA", "TUBB3", "DOK5", "NPY", "HES5",
]

with plt.rc_context({"figure.dpi": 300}):
    ax = sc.pl.heatmap(
        adata_morph_rpnc_sorted,
        var_names=rpnc_genes,
        groupby="leiden_morph_rpnc",
        swap_axes=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )
    bold_selected_heatmap_yticklabels(ax, rpnc_bold_genes)
    ax["groupby_ax"].set_xlabel("")
    plt.savefig(figures_dir / "rpnc_key_gene_heatmap_bead.png", bbox_inches="tight", pad_inches=0)
    plt.savefig(figures_dir / "rpnc_key_gene_heatmap_bead.pdf", bbox_inches="tight", pad_inches=0)
    plt.savefig(MANUSCRIPT_FIG_DIR / "Fig4d_heatmap_rpnc_key_genes_bead.png", bbox_inches="tight", pad_inches=0)
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    fig_dict = sc.pl.heatmap(
        adata_morph_rpnc_sorted,
        var_names=rpnc_genes,
        groupby="leiden_morph_rpnc",
        swap_axes=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )
    hide_heatmap_axes_and_colorbar(fig_dict)
    plt.savefig(figures_dir / "rpnc_key_gene_heatmap_bead_noaxes.png", bbox_inches="tight", pad_inches=0)
    plt.savefig(MANUSCRIPT_FIG_DIR / "Fig4d_heatmap_rpnc_key_genes_bead_noaxes.png", bbox_inches="tight", pad_inches=0)
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    ax = plot_empty_heatmap_axes(
        adata_morph_rpnc_sorted,
        var_names=rpnc_genes,
        groupby="leiden_morph_rpnc",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    keep_only_selected_heatmap_yticklabels(ax, rpnc_bold_genes)
    ax["groupby_ax"].set_xlabel("")
    plt.savefig(figures_dir / "rpnc_key_gene_heatmap_bead_axesonly.svg", bbox_inches="tight", pad_inches=0)
    plt.savefig(figures_dir / "rpnc_key_gene_heatmap_bead_axesonly.pdf", bbox_inches="tight", pad_inches=0)
    plt.savefig(MANUSCRIPT_FIG_DIR / "Fig4d_heatmap_rpnc_key_genes_bead_axesonly.svg", bbox_inches="tight", pad_inches=0)
    plt.savefig(MANUSCRIPT_FIG_DIR / "Fig4d_heatmap_rpnc_key_genes_bead_axesonly.pdf", bbox_inches="tight", pad_inches=0)
    plt.show()

for _adata in [
    adata_morph_nc_, adata_morph_nc_SMD_, adata_morph_rp_, adata_morph_rp_SMD_,
    adata_morph_rpnc_, adata_morph_rpnc_filtered, adata_morph_rpnc_SMD, adata_morph_rpnc_SMD_,
]:
    assert_gene_id_index(_adata)
save_h5ad(adata_morph_nc_, stage_path / 'adata_morph_nc_.h5ad')
save_h5ad(adata_morph_nc_SMD_, stage_path / 'adata_morph_nc_SMD_.h5ad')
save_h5ad(adata_morph_rp_, stage_path / 'adata_morph_rp_.h5ad')
save_h5ad(adata_morph_rp_SMD_, stage_path / 'adata_morph_rp_SMD_.h5ad')
save_h5ad(adata_morph_rpnc_, stage_path / 'adata_morph_rpnc.h5ad')
save_h5ad(adata_morph_rpnc_filtered, stage_path / 'adata_morph_rpnc_pre_smd_.h5ad')
save_h5ad(adata_morph_rpnc_SMD, stage_path / 'adata_morph_rpnc_SMD.h5ad')
save_h5ad(adata_morph_rpnc_SMD_, stage_path / 'adata_morph_rpnc_SMD_.h5ad')

rpnc_meta = json.loads((stage_path / "meta.json").read_text())
rpnc_meta.update(
    {
        "status": "default_labels_adopted_from_leiden_ensemble",
        "cluster_key": "leiden_morph_rpnc",
        "chosen_leiden_resolution": chosen_rpnc_resolution,
        "chosen_leiden_resolution_display": f"{chosen_rpnc_resolution:.2f}",
        "ensemble_default_label_source": "leiden_ensemble/results/intermediates/06_rp_nc_subclustering",
        "chosen_cluster_label_style": "suffix_stripped" if use_base_labels else "raw_with_suffix",
        "n_clusters_default": int(len(cluster_label_order)),
        "figures_dir": "figures",
        "tables_dir": "tables",
    }
)
save_json(rpnc_meta, stage_path / "meta.json")

print(
    f"Adopted leiden_ensemble/06 labels at resolution {chosen_rpnc_resolution:.2f} and saved RP/NC figures/tables to {stage_path}"
)
